In [0]:
# MAGIC %md
# MAGIC # Unidad 3. Evidencia de Aprendizaje EA3  
# MAGIC ## Taller: procesamiento de datos en una infraestructura cloud con Databricks, Spark y SQL
# MAGIC
# MAGIC **Estudiante:** Pablo Emilio Gómez Gómez  
# MAGIC **Programa:** Ingeniería de Software y Datos  
# MAGIC **Asignatura:** Big Data  
# MAGIC **Fecha:** 24 de mayo de 2026  
# MAGIC **Notebook:** Gomez_Pablo_Actividad_2  
# MAGIC
# MAGIC ---
# MAGIC
# MAGIC ## 1. Contexto de la actividad
# MAGIC
# MAGIC Esta evidencia de aprendizaje desarrolla un ejercicio práctico de procesamiento de datos sobre una infraestructura cloud utilizando **Databricks Community Edition / Free Edition**, **Apache Spark**, **PySpark** y **Spark SQL**.
# MAGIC
# MAGIC El propósito central es tomar un conjunto de datos proveniente de Kaggle, diseñar un esquema de almacenamiento, cargar los datos en Databricks, crear una tabla consultable y validar su procesamiento mediante operaciones equivalentes en Spark y SQL.
# MAGIC
# MAGIC Para el ejercicio se trabajará con el dataset **Titanic**, ampliamente utilizado en prácticas académicas de analítica de datos porque contiene variables demográficas, sociales y de supervivencia de pasajeros. Su estructura permite realizar validaciones de conteo, descripción, selección, agrupación y comparación entre formas de consulta.

In [0]:
# ============================================================
# CELDA 2 - CONFIGURACIÓN DEL ENTORNO DATABRICKS
# Versión compatible con Databricks Serverless
# ============================================================

# Cómo se lee:
#   Importamos librerías básicas para consultar fecha, sistema operativo
#   y versión de Python.
# Para qué sirve:
#   Estas salidas sirven como evidencia de configuración del entorno
#   solicitado en la rúbrica de la EA3.

from datetime import datetime
from zoneinfo import ZoneInfo
import sys
import platform

# Cómo se lee:
#   Se toma la fecha y hora actual en la zona horaria de Bogotá.
# Para qué sirve:
#   Permite dejar evidencia del momento real en que se ejecutó el notebook.
fecha_bogota = datetime.now(ZoneInfo("America/Bogota")).strftime("%Y-%m-%d %H:%M:%S")

print("=" * 70)
print("CONFIGURACIÓN GENERAL DEL ENTORNO")
print("=" * 70)

print(f"Fecha y hora de ejecución en Bogotá: {fecha_bogota}")
print(f"Versión de Python: {sys.version}")
print(f"Sistema operativo reportado: {platform.platform()}")

print("\n" + "=" * 70)
print("CONFIGURACIÓN DE APACHE SPARK")
print("=" * 70)

# Cómo se lee:
#   spark.version consulta la versión de Apache Spark disponible.
# Para qué sirve:
#   Evidencia que el notebook está trabajando sobre Spark dentro de Databricks.
print(f"Versión de Spark: {spark.version}")

print("\n" + "=" * 70)
print("CONFIGURACIÓN DE DATABRICKS / SQL")
print("=" * 70)

# Cómo se lee:
#   Se consulta el catálogo y esquema actual usando Spark SQL.
# Para qué sirve:
#   Muestra el contexto de trabajo donde se crearán bases, tablas y consultas.
spark.sql("""
SELECT
  current_catalog() AS catalogo_actual,
  current_schema() AS esquema_actual,
  current_database() AS base_datos_actual
""").show(truncate=False)

print("\n" + "=" * 70)
print("CONFIGURACIONES DISPONIBLES SIN USAR SPARKCONTEXT")
print("=" * 70)

# Cómo se lee:
#   SET -v lista configuraciones visibles desde Spark SQL.
# Para qué sirve:
#   Reemplaza spark.sparkContext.getConf().getAll(), porque en Serverless
#   Databricks no permite acceder directamente al SparkContext.
config_df = spark.sql("SET -v")

# Cómo se lee:
#   Filtramos solo algunas configuraciones relacionadas con Spark y Databricks.
# Para qué sirve:
#   Evita imprimir demasiada información y deja una evidencia clara.
config_df.filter(
    "lower(key) LIKE '%spark%' OR lower(key) LIKE '%databricks%'"
).select(
    "key", "value", "meaning"
).show(30, truncate=False)

print("\n" + "=" * 70)
print("VALIDACIÓN TERMINADA")
print("=" * 70)

print("La configuración fue consultada correctamente sin usar spark.sparkContext.")
print("Esto es compatible con Databricks Serverless y sirve como evidencia del entorno.")

CONFIGURACIÓN GENERAL DEL ENTORNO
Fecha y hora de ejecución en Bogotá: 2026-05-24 22:54:29
Versión de Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Sistema operativo reportado: Linux-6.1.170-210.320.amzn2023.aarch64-aarch64-with-glibc2.39

CONFIGURACIÓN DE APACHE SPARK
Versión de Spark: 4.1.0

CONFIGURACIÓN DE DATABRICKS / SQL
+---------------+--------------+-----------------+
|catalogo_actual|esquema_actual|base_datos_actual|
+---------------+--------------+-----------------+
|workspace      |default       |default          |
+---------------+--------------+-----------------+


CONFIGURACIONES DISPONIBLES SIN USAR SPARKCONTEXT
+----------------------------------+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 3. Diseño del esquema de datos

Para esta evidencia se trabajará con el conjunto de datos Titanic, un dataset clásico de análisis de datos disponible en Kaggle. Este conjunto permite estudiar variables asociadas a pasajeros, características sociodemográficas, clase del tiquete, tarifa pagada, puerto de embarque y supervivencia.

La elección de este dataset es adecuada para el ejercicio porque tiene una estructura tabular clara, permite definir un esquema de almacenamiento en Spark, facilita consultas SQL y también permite realizar validaciones con PySpark mediante conteos, filtros, agrupaciones y descripciones estadísticas.

## Entidad principal

La entidad principal del modelo es pasajero_titanic. Cada registro representa un pasajero del Titanic.

## Diccionario de datos preliminar

| Campo | Tipo de dato | Llave | Nulabilidad | Descripción |
|---|---|---|---|---|
| PassengerId | Integer | Primaria | No nulo | Identificador único del pasajero. |
| Survived | Integer | No aplica | No nulo | Variable objetivo: 0 = no sobrevivió, 1 = sobrevivió. |
| Pclass | Integer | No aplica | No nulo | Clase del tiquete: 1, 2 o 3. |
| Name | String | No aplica | No nulo | Nombre completo del pasajero. |
| Sex | String | No aplica | No nulo | Sexo registrado del pasajero. |
| Age | Double | No aplica | Puede ser nulo | Edad del pasajero. |
| SibSp | Integer | No aplica | No nulo | Número de hermanos o cónyuge a bordo. |
| Parch | Integer | No aplica | No nulo | Número de padres o hijos a bordo. |
| Ticket | String | No aplica | No nulo | Número o código del tiquete. |
| Fare | Double | No aplica | No nulo | Tarifa pagada por el pasajero. |
| Cabin | String | No aplica | Puede ser nulo | Cabina asignada al pasajero. |
| Embarked | String | No aplica | Puede ser nulo | Puerto de embarque: C, Q o S. |

## Diagrama simple del esquema

pasajero_titanic  
│  
├── PassengerId: identificador único del pasajero  
├── Survived: estado de supervivencia  
├── Pclass: clase del tiquete  
├── Name: nombre del pasajero  
├── Sex: sexo registrado  
├── Age: edad  
├── SibSp: hermanos o cónyuge a bordo  
├── Parch: padres o hijos a bordo  
├── Ticket: código del tiquete  
├── Fare: tarifa pagada  
├── Cabin: cabina asignada  
└── Embarked: puerto de embarque  

Este diseño permite cumplir el primer criterio de la rúbrica, porque presenta el esquema de datos con tipos, llave primaria, nulabilidad, descripción de campos y una representación visual simple.


In [0]:
# ============================================================
# CELDA 4 - DEFINICIÓN TÉCNICA DEL ESQUEMA EN PYSPARK
# ============================================================

# Cómo se lee:
#   Importamos los tipos de datos de PySpark.
# Para qué sirve:
#   Permite construir un esquema formal para leer el dataset
#   sin depender solamente de la inferencia automática de Spark.

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)

print("=" * 70)
print("DEFINICIÓN DEL ESQUEMA TITANIC EN PYSPARK")
print("=" * 70)

# Cómo se lee:
#   StructType representa la estructura completa de la tabla.
#   StructField representa cada columna, su tipo de dato y si acepta nulos.
# Para qué sirve:
#   Este esquema será usado después en spark.read.csv()
#   para cargar el archivo de forma controlada.

schema_titanic = StructType([
    StructField("PassengerId", IntegerType(), False),
    StructField("Survived", IntegerType(), False),
    StructField("Pclass", IntegerType(), False),
    StructField("Name", StringType(), False),
    StructField("Sex", StringType(), False),
    StructField("Age", DoubleType(), True),
    StructField("SibSp", IntegerType(), False),
    StructField("Parch", IntegerType(), False),
    StructField("Ticket", StringType(), False),
    StructField("Fare", DoubleType(), False),
    StructField("Cabin", StringType(), True),
    StructField("Embarked", StringType(), True)
])

print("Esquema PySpark creado correctamente.")
print()
print(schema_titanic.simpleString())

print("\n" + "=" * 70)
print("DDL PROPUESTO EN SPARK SQL")
print("=" * 70)

# Cómo se lee:
#   Este bloque representa cómo se podría crear la tabla usando SQL.
# Para qué sirve:
#   Cumple la parte de la rúbrica que pide DDL o StructType.

ddl_titanic = """
CREATE TABLE IF NOT EXISTS ea3_bigdata.titanic_pasajeros (
    PassengerId INT NOT NULL,
    Survived INT NOT NULL,
    Pclass INT NOT NULL,
    Name STRING NOT NULL,
    Sex STRING NOT NULL,
    Age DOUBLE,
    SibSp INT NOT NULL,
    Parch INT NOT NULL,
    Ticket STRING NOT NULL,
    Fare DOUBLE NOT NULL,
    Cabin STRING,
    Embarked STRING
)
USING DELTA;
"""

print(ddl_titanic)

print("=" * 70)
print("CREACIÓN DEL CATÁLOGO LÓGICO DE TRABAJO")
print("=" * 70)

# Cómo se lee:
#   Se indica que trabajaremos dentro del catálogo workspace.
# Para qué sirve:
#   Ordena el ambiente de trabajo dentro de Databricks.

spark.sql("USE CATALOG workspace")

# Cómo se lee:
#   Creamos un esquema o base de datos llamado ea3_bigdata.
# Para qué sirve:
#   Allí se guardará la tabla Delta que usaremos en la actividad.

spark.sql("CREATE SCHEMA IF NOT EXISTS ea3_bigdata")

# Cómo se lee:
#   Seleccionamos el esquema recién creado.
# Para qué sirve:
#   Las tablas siguientes se crearán dentro de ea3_bigdata.

spark.sql("USE SCHEMA ea3_bigdata")

print("Catálogo seleccionado: workspace")
print("Esquema/base de datos creado y seleccionado: ea3_bigdata")

print("\n" + "=" * 70)
print("DICCIONARIO DE DATOS COMO TABLA SPARK")
print("=" * 70)

# Cómo se lee:
#   Creamos una lista de diccionarios con los campos del dataset.
# Para qué sirve:
#   Permite mostrar el diccionario de datos como una tabla dentro del notebook.

diccionario_datos = [
    {"campo": "PassengerId", "tipo": "Integer", "llave": "Primaria", "nulabilidad": "No nulo", "descripcion": "Identificador único del pasajero."},
    {"campo": "Survived", "tipo": "Integer", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "0 = no sobrevivió, 1 = sobrevivió."},
    {"campo": "Pclass", "tipo": "Integer", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "Clase del tiquete: 1, 2 o 3."},
    {"campo": "Name", "tipo": "String", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "Nombre completo del pasajero."},
    {"campo": "Sex", "tipo": "String", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "Sexo registrado del pasajero."},
    {"campo": "Age", "tipo": "Double", "llave": "No aplica", "nulabilidad": "Puede ser nulo", "descripcion": "Edad del pasajero."},
    {"campo": "SibSp", "tipo": "Integer", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "Hermanos o cónyuge a bordo."},
    {"campo": "Parch", "tipo": "Integer", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "Padres o hijos a bordo."},
    {"campo": "Ticket", "tipo": "String", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "Código o número del tiquete."},
    {"campo": "Fare", "tipo": "Double", "llave": "No aplica", "nulabilidad": "No nulo", "descripcion": "Tarifa pagada por el pasajero."},
    {"campo": "Cabin", "tipo": "String", "llave": "No aplica", "nulabilidad": "Puede ser nulo", "descripcion": "Cabina asignada."},
    {"campo": "Embarked", "tipo": "String", "llave": "No aplica", "nulabilidad": "Puede ser nulo", "descripcion": "Puerto de embarque."}
]

df_diccionario = spark.createDataFrame(diccionario_datos)

display(df_diccionario)

print("\n" + "=" * 70)
print("VALIDACIÓN TERMINADA")
print("=" * 70)
print("El esquema de datos quedó definido en Markdown, PySpark y DDL Spark SQL.")
print("También quedó creado el espacio lógico de trabajo para la tabla de la EA3.")

DEFINICIÓN DEL ESQUEMA TITANIC EN PYSPARK
Esquema PySpark creado correctamente.

struct<PassengerId:int,Survived:int,Pclass:int,Name:string,Sex:string,Age:double,SibSp:int,Parch:int,Ticket:string,Fare:double,Cabin:string,Embarked:string>

DDL PROPUESTO EN SPARK SQL

CREATE TABLE IF NOT EXISTS ea3_bigdata.titanic_pasajeros (
    PassengerId INT NOT NULL,
    Survived INT NOT NULL,
    Pclass INT NOT NULL,
    Name STRING NOT NULL,
    Sex STRING NOT NULL,
    Age DOUBLE,
    SibSp INT NOT NULL,
    Parch INT NOT NULL,
    Ticket STRING NOT NULL,
    Fare DOUBLE NOT NULL,
    Cabin STRING,
    Embarked STRING
)
USING DELTA;

CREACIÓN DEL CATÁLOGO LÓGICO DE TRABAJO
Catálogo seleccionado: workspace
Esquema/base de datos creado y seleccionado: ea3_bigdata

DICCIONARIO DE DATOS COMO TABLA SPARK


campo,descripcion,llave,nulabilidad,tipo
PassengerId,Identificador único del pasajero.,Primaria,No nulo,Integer
Survived,"0 = no sobrevivió, 1 = sobrevivió.",No aplica,No nulo,Integer
Pclass,"Clase del tiquete: 1, 2 o 3.",No aplica,No nulo,Integer
Name,Nombre completo del pasajero.,No aplica,No nulo,String
Sex,Sexo registrado del pasajero.,No aplica,No nulo,String
Age,Edad del pasajero.,No aplica,Puede ser nulo,Double
SibSp,Hermanos o cónyuge a bordo.,No aplica,No nulo,Integer
Parch,Padres o hijos a bordo.,No aplica,No nulo,Integer
Ticket,Código o número del tiquete.,No aplica,No nulo,String
Fare,Tarifa pagada por el pasajero.,No aplica,No nulo,Double



VALIDACIÓN TERMINADA
El esquema de datos quedó definido en Markdown, PySpark y DDL Spark SQL.
También quedó creado el espacio lógico de trabajo para la tabla de la EA3.


# 4. Obtención, ingesta y creación de tabla

En esta sección se realiza la carga del dataset Titanic, utilizado como conjunto de datos de referencia para el procesamiento en Databricks. La fuente corresponde al dataset Titanic disponible públicamente para ejercicios académicos de ciencia de datos y análisis con Spark.

La ingesta se realizará desde una URL pública del archivo CSV, con el propósito de facilitar la reproducibilidad del ejercicio dentro de Databricks Community/Free Edition sin depender de credenciales privadas de Kaggle. Posteriormente, los datos serán leídos con el esquema definido en PySpark, convertidos en un DataFrame de Spark y persistidos como tabla Delta en el esquema `workspace.ea3_bigdata`.

Esta parte permite evidenciar tres elementos solicitados por la actividad: obtención de datos, carga en el entorno cloud y creación de una tabla procesable con Spark y SQL.

In [0]:
# ============================================================
# CELDA 5 - INGESTA DEL DATASET TITANIC Y CREACIÓN DE TABLA
# ============================================================

# Cómo se lee:
#   Importamos pandas y los tipos de datos de PySpark.
# Para qué sirve:
#   Pandas ayuda a traer el CSV desde una URL pública.
#   PySpark permite definir el esquema formal del DataFrame.

import pandas as pd

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)

print("=" * 70)
print("INGESTA DEL DATASET TITANIC")
print("=" * 70)

# Cómo se lee:
#   Guardamos en una variable la URL pública del archivo CSV.
# Para qué sirve:
#   Permite traer los datos al notebook de Databricks
#   sin descargar manualmente el archivo en el computador.

url_titanic = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

print("Fuente del archivo CSV:")
print(url_titanic)

# Cómo se lee:
#   Leemos el archivo CSV usando pandas.
# Para qué sirve:
#   Convierte el archivo en una tabla temporal en memoria
#   para luego transformarla en DataFrame de Spark.

pdf_titanic = pd.read_csv(url_titanic)

print("\nArchivo leído correctamente con pandas.")
print(f"Filas cargadas inicialmente: {len(pdf_titanic)}")
print(f"Columnas encontradas: {list(pdf_titanic.columns)}")

# Cómo se lee:
#   Convertimos valores vacíos o NaN a None.
# Para qué sirve:
#   Spark maneja mejor los valores nulos cuando llegan como None.

pdf_titanic = pdf_titanic.where(pd.notnull(pdf_titanic), None)

# Cómo se lee:
#   Ajustamos los tipos numéricos principales.
# Para qué sirve:
#   Evita errores al crear el DataFrame con el esquema definido.

pdf_titanic["PassengerId"] = pdf_titanic["PassengerId"].astype(int)
pdf_titanic["Survived"] = pdf_titanic["Survived"].astype(int)
pdf_titanic["Pclass"] = pdf_titanic["Pclass"].astype(int)
pdf_titanic["SibSp"] = pdf_titanic["SibSp"].astype(int)
pdf_titanic["Parch"] = pdf_titanic["Parch"].astype(int)
pdf_titanic["Age"] = pd.to_numeric(pdf_titanic["Age"], errors="coerce")
pdf_titanic["Fare"] = pd.to_numeric(pdf_titanic["Fare"], errors="coerce")

# Cómo se lee:
#   Definimos nuevamente el esquema Titanic.
# Para qué sirve:
#   Garantiza que esta celda pueda ejecutarse aunque el notebook
#   se reinicie o se pierdan variables anteriores.

schema_titanic = StructType([
    StructField("PassengerId", IntegerType(), False),
    StructField("Survived", IntegerType(), False),
    StructField("Pclass", IntegerType(), False),
    StructField("Name", StringType(), False),
    StructField("Sex", StringType(), False),
    StructField("Age", DoubleType(), True),
    StructField("SibSp", IntegerType(), False),
    StructField("Parch", IntegerType(), False),
    StructField("Ticket", StringType(), False),
    StructField("Fare", DoubleType(), False),
    StructField("Cabin", StringType(), True),
    StructField("Embarked", StringType(), True)
])

# Cómo se lee:
#   Creamos un DataFrame de Spark a partir de pandas.
# Para qué sirve:
#   A partir de este punto, los datos ya se procesan con Spark.

df_titanic = spark.createDataFrame(pdf_titanic, schema=schema_titanic)

print("\nDataFrame de Spark creado correctamente.")

print("\n" + "=" * 70)
print("ESQUEMA DEL DATAFRAME")
print("=" * 70)

df_titanic.printSchema()

# Cómo se lee:
#   Seleccionamos el catálogo workspace y el esquema ea3_bigdata.
# Para qué sirve:
#   Organiza la tabla dentro del entorno de Databricks.

spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS ea3_bigdata")
spark.sql("USE SCHEMA ea3_bigdata")

# Cómo se lee:
#   Guardamos el DataFrame como tabla Delta.
# Para qué sirve:
#   Permite consultar los datos posteriormente con SQL y Spark.

df_titanic.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ea3_bigdata.titanic_pasajeros")

print("\nTabla Delta creada correctamente:")
print("workspace.ea3_bigdata.titanic_pasajeros")

print("\n" + "=" * 70)
print("VALIDACIONES INICIALES")
print("=" * 70)

# Cómo se lee:
#   Contamos cuántas filas tiene el DataFrame.
# Para qué sirve:
#   Verifica que los datos fueron cargados.

total_filas = df_titanic.count()
total_columnas = len(df_titanic.columns)

print(f"Total de filas cargadas: {total_filas}")
print(f"Total de columnas cargadas: {total_columnas}")

print("\nPrimeras 10 filas del dataset:")
display(df_titanic.limit(10))

print("\nConteo por sexo:")
display(df_titanic.groupBy("Sex").count())

print("\nConteo por clase del tiquete:")
display(df_titanic.groupBy("Pclass").count())

print("\n" + "=" * 70)
print("PROCESO DE INGESTA FINALIZADO")
print("=" * 70)
print("El dataset fue cargado, convertido a DataFrame de Spark y guardado como tabla Delta.")

INGESTA DEL DATASET TITANIC
Fuente del archivo CSV:
https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

Archivo leído correctamente con pandas.
Filas cargadas inicialmente: 891
Columnas encontradas: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

DataFrame de Spark creado correctamente.

ESQUEMA DEL DATAFRAME
root
 |-- PassengerId: integer (nullable = false)
 |-- Survived: integer (nullable = false)
 |-- Pclass: integer (nullable = false)
 |-- Name: string (nullable = false)
 |-- Sex: string (nullable = false)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = false)
 |-- Parch: integer (nullable = false)
 |-- Ticket: string (nullable = false)
 |-- Fare: double (nullable = false)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)


Tabla Delta creada correctamente:
workspace.ea3_bigdata.titanic_pasajeros

VALIDACIONES INICIALES
Total de filas cargad

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,null,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,null,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,null,S
6,0,3,"Moran, Mr. James",male,null,0,0,330877,8.4583,null,Q
7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,null,S
9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,null,S
10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,null,C



Conteo por sexo:


Sex,count
male,577
female,314



Conteo por clase del tiquete:


Pclass,count
3,491
1,216
2,184



PROCESO DE INGESTA FINALIZADO
El dataset fue cargado, convertido a DataFrame de Spark y guardado como tabla Delta.


# 5. Validaciones con Spark y SQL

En esta sección se valida que la tabla `workspace.ea3_bigdata.titanic_pasajeros` haya quedado correctamente creada y disponible para procesamiento analítico. Para ello se realizan verificaciones desde dos enfoques: primero con PySpark, usando operaciones sobre DataFrames, y luego con Spark SQL, mediante consultas declarativas.

Las validaciones incluyen revisión del esquema, conteo de registros, identificación de valores nulos, descripción estadística de variables numéricas, consultas de selección, agrupación y filtros. Estos pasos permiten comprobar que los datos fueron ingeridos correctamente, que la tabla puede ser consultada desde Databricks y que el entorno cloud permite procesar información tanto con Spark como con SQL.

In [0]:
# ============================================================
# CELDA 6 - VALIDACIONES CON PYSPARK
# ============================================================

# Cómo se lee:
#   Importamos funciones de PySpark para contar, promediar,
#   redondear, filtrar y revisar valores nulos.
# Para qué sirve:
#   Estas funciones permiten validar el contenido de la tabla
#   usando operaciones propias de Spark DataFrame.

from pyspark.sql.functions import (
    col,
    count,
    avg,
    round as spark_round,
    sum as spark_sum,
    when
)

print("=" * 70)
print("VALIDACIONES CON PYSPARK")
print("=" * 70)

# Cómo se lee:
#   Guardamos el nombre completo de la tabla en una variable.
# Para qué sirve:
#   Evita repetir el nombre completo de la tabla en cada consulta.

nombre_tabla = "workspace.ea3_bigdata.titanic_pasajeros"

# Cómo se lee:
#   Leemos la tabla Delta creada anteriormente.
# Para qué sirve:
#   Permite validar que la tabla existe y que Spark puede consultarla.

df_validacion = spark.table(nombre_tabla)

print("\nTabla cargada desde el catálogo de Databricks:")
print(nombre_tabla)

print("\n" + "=" * 70)
print("1. ESQUEMA DE LA TABLA")
print("=" * 70)

# Cómo se lee:
#   Imprimimos la estructura de columnas y tipos de datos.
# Para qué sirve:
#   Verifica que el esquema aplicado corresponde al diseño definido.

df_validacion.printSchema()

print("\n" + "=" * 70)
print("2. CONTEO GENERAL DE REGISTROS Y COLUMNAS")
print("=" * 70)

# Cómo se lee:
#   Contamos filas y columnas.
# Para qué sirve:
#   Confirma la cantidad total de datos disponibles para análisis.

total_filas = df_validacion.count()
total_columnas = len(df_validacion.columns)

print(f"Total de filas: {total_filas}")
print(f"Total de columnas: {total_columnas}")

print("\n" + "=" * 70)
print("3. MUESTRA DE REGISTROS")
print("=" * 70)

# Cómo se lee:
#   Mostramos las primeras 10 filas.
# Para qué sirve:
#   Permite revisar visualmente que los datos se cargaron bien.

display(df_validacion.limit(10))

print("\n" + "=" * 70)
print("4. VALIDACIÓN DE VALORES NULOS POR COLUMNA")
print("=" * 70)

# Cómo se lee:
#   Recorremos cada columna y contamos cuántos valores nulos tiene.
# Para qué sirve:
#   Permite identificar campos incompletos, como Age, Cabin o Embarked.

df_nulos = df_validacion.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_validacion.columns
])

display(df_nulos)

print("\n" + "=" * 70)
print("5. DESCRIPCIÓN ESTADÍSTICA DE VARIABLES NUMÉRICAS")
print("=" * 70)

# Cómo se lee:
#   Calculamos estadísticas básicas de variables numéricas.
# Para qué sirve:
#   Permite revisar conteo, promedio, desviación, mínimo y máximo.

display(
    df_validacion
    .select("Age", "Fare", "SibSp", "Parch")
    .describe()
)

print("\n" + "=" * 70)
print("6. AGRUPACIÓN POR SEXO")
print("=" * 70)

# Cómo se lee:
#   Agrupamos por sexo y calculamos conteo y tasa promedio de supervivencia.
# Para qué sirve:
#   Permite analizar diferencias generales de supervivencia por grupo.

df_por_sexo = (
    df_validacion
    .groupBy("Sex")
    .agg(
        count("*").alias("total_pasajeros"),
        spark_round(avg("Survived") * 100, 2).alias("porcentaje_supervivencia")
    )
    .orderBy("Sex")
)

display(df_por_sexo)

print("\n" + "=" * 70)
print("7. AGRUPACIÓN POR CLASE DEL TIQUETE")
print("=" * 70)

# Cómo se lee:
#   Agrupamos por clase del tiquete.
# Para qué sirve:
#   Permite comparar cantidad de pasajeros y supervivencia por clase.

df_por_clase = (
    df_validacion
    .groupBy("Pclass")
    .agg(
        count("*").alias("total_pasajeros"),
        spark_round(avg("Survived") * 100, 2).alias("porcentaje_supervivencia"),
        spark_round(avg("Fare"), 2).alias("tarifa_promedio")
    )
    .orderBy("Pclass")
)

display(df_por_clase)

print("\n" + "=" * 70)
print("8. FILTRO DE PASAJEROS SOBREVIVIENTES CON TARIFA MAYOR A 50")
print("=" * 70)

# Cómo se lee:
#   Filtramos pasajeros que sobrevivieron y pagaron una tarifa mayor a 50.
# Para qué sirve:
#   Muestra cómo Spark permite hacer filtros analíticos sobre el dataset.

df_filtro = (
    df_validacion
    .filter((col("Survived") == 1) & (col("Fare") > 50))
    .select("PassengerId", "Name", "Sex", "Age", "Pclass", "Fare", "Embarked")
    .orderBy(col("Fare").desc())
)

display(df_filtro.limit(10))

print("\n" + "=" * 70)
print("VALIDACIONES PYSPARK FINALIZADAS")
print("=" * 70)
print("Se validó esquema, conteo, nulos, descripción estadística, agrupaciones y filtros con Spark DataFrame.")

VALIDACIONES CON PYSPARK

Tabla cargada desde el catálogo de Databricks:
workspace.ea3_bigdata.titanic_pasajeros

1. ESQUEMA DE LA TABLA
root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)


2. CONTEO GENERAL DE REGISTROS Y COLUMNAS
Total de filas: 891
Total de columnas: 12

3. MUESTRA DE REGISTROS


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,null,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,null,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,null,S
6,0,3,"Moran, Mr. James",male,null,0,0,330877,8.4583,null,Q
7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,null,S
9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,null,S
10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,null,C



4. VALIDACIÓN DE VALORES NULOS POR COLUMNA


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,0,0,0,177,0,0,0,0,687,2



5. DESCRIPCIÓN ESTADÍSTICA DE VARIABLES NUMÉRICAS


...


7. AGRUPACIÓN POR CLASE DEL TIQUETE


Pclass,total_pasajeros,porcentaje_supervivencia,tarifa_promedio
1,216,62.96,84.15
2,184,47.28,20.66
3,491,24.24,13.68



8. FILTRO DE PASAJEROS SOBREVIVIENTES CON TARIFA MAYOR A 50


PassengerId,Name,Sex,Age,Pclass,Fare,Embarked
259,"Ward, Miss. Anna",female,35.0,1,512.3292,C
680,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,1,512.3292,C
738,"Lesurer, Mr. Gustave J",male,35.0,1,512.3292,C
89,"Fortune, Miss. Mabel Helen",female,23.0,1,263.0,S
342,"Fortune, Miss. Alice Elizabeth",female,24.0,1,263.0,S
312,"Ryerson, Miss. Emily Borie",female,18.0,1,262.375,C
743,"Ryerson, Miss. Susan Parker ""Suzette""",female,21.0,1,262.375,C
300,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",female,50.0,1,247.5208,C
381,"Bidois, Miss. Rosalie",female,42.0,1,227.525,C
717,"Endres, Miss. Caroline Louise",female,38.0,1,227.525,C



VALIDACIONES PYSPARK FINALIZADAS
Se validó esquema, conteo, nulos, descripción estadística, agrupaciones y filtros con Spark DataFrame.


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA ea3_bigdata;

DESCRIBE TABLE titanic_pasajeros;

col_name,data_type,comment
PassengerId,int,null
Survived,int,null
Pclass,int,null
Name,string,null
Sex,string,null
Age,double,null
SibSp,int,null
Parch,int,null
Ticket,string,null
Fare,double,null


In [0]:
%sql
SHOW CREATE TABLE workspace.ea3_bigdata.titanic_pasajeros;

createtab_stmt
"CREATE TABLE workspace.ea3_bigdata.titanic_pasajeros ( PassengerId INT, Survived INT, Pclass INT, Name STRING COLLATE UTF8_BINARY, Sex STRING COLLATE UTF8_BINARY, Age DOUBLE, SibSp INT, Parch INT, Ticket STRING COLLATE UTF8_BINARY, Fare DOUBLE, Cabin STRING COLLATE UTF8_BINARY, Embarked STRING COLLATE UTF8_BINARY) USING delta TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


In [0]:
%sql
SELECT
  COUNT(*) AS total_registros,
  COUNT(DISTINCT PassengerId) AS pasajeros_unicos,
  MIN(Age) AS edad_minima,
  MAX(Age) AS edad_maxima,
  ROUND(AVG(Fare), 2) AS tarifa_promedio
FROM workspace.ea3_bigdata.titanic_pasajeros;

total_registros,pasajeros_unicos,edad_minima,edad_maxima,tarifa_promedio
891,891,0.42,80.0,32.2


In [0]:
%sql
SELECT
  Sex,
  COUNT(*) AS total_pasajeros,
  ROUND(AVG(Survived) * 100, 2) AS porcentaje_supervivencia
FROM workspace.ea3_bigdata.titanic_pasajeros
GROUP BY Sex
ORDER BY Sex;

Sex,total_pasajeros,porcentaje_supervivencia
female,314,74.2
male,577,18.89


In [0]:
%sql
SELECT
  Pclass,
  COUNT(*) AS total_pasajeros,
  ROUND(AVG(Survived) * 100, 2) AS porcentaje_supervivencia,
  ROUND(AVG(Fare), 2) AS tarifa_promedio
FROM workspace.ea3_bigdata.titanic_pasajeros
GROUP BY Pclass
ORDER BY Pclass;

Pclass,total_pasajeros,porcentaje_supervivencia,tarifa_promedio
1,216,62.96,84.15
2,184,47.28,20.66
3,491,24.24,13.68


In [0]:
%sql
SELECT
  PassengerId,
  Name,
  Sex,
  Age,
  Pclass,
  Fare,
  Embarked
FROM workspace.ea3_bigdata.titanic_pasajeros
WHERE Survived = 1
  AND Fare > 50
ORDER BY Fare DESC
LIMIT 10;

PassengerId,Name,Sex,Age,Pclass,Fare,Embarked
259,"Ward, Miss. Anna",female,35.0,1,512.3292,C
680,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,1,512.3292,C
738,"Lesurer, Mr. Gustave J",male,35.0,1,512.3292,C
89,"Fortune, Miss. Mabel Helen",female,23.0,1,263.0,S
342,"Fortune, Miss. Alice Elizabeth",female,24.0,1,263.0,S
312,"Ryerson, Miss. Emily Borie",female,18.0,1,262.375,C
743,"Ryerson, Miss. Susan Parker ""Suzette""",female,21.0,1,262.375,C
300,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",female,50.0,1,247.5208,C
381,"Bidois, Miss. Rosalie",female,42.0,1,227.525,C
717,"Endres, Miss. Caroline Louise",female,38.0,1,227.525,C


# 6. Comparación entre SQL y Spark

En esta práctica se trabajó el mismo conjunto de datos desde dos formas de procesamiento: PySpark, mediante operaciones sobre DataFrames, y Spark SQL, mediante consultas declarativas. Ambos enfoques permitieron obtener los mismos resultados generales, pero cada uno ofrece ventajas distintas según el tipo de tarea.

SQL resultó más directo para consultas de lectura, conteos, filtros y agrupaciones. Su sintaxis es clara, cercana al lenguaje usado tradicionalmente en bases de datos relacionales y facilita validar rápidamente la información cargada en la tabla Delta. En cambio, PySpark ofreció mayor flexibilidad para construir transformaciones paso a paso, encadenar operaciones y preparar análisis más complejos dentro del mismo notebook.

A partir del ejercicio realizado, puede afirmarse que SQL es especialmente útil cuando se desea consultar y resumir datos de manera rápida, mientras que Spark con PySpark resulta más conveniente cuando se requiere construir procesos de transformación, limpieza, validación o análisis más elaborados sobre grandes volúmenes de información.

In [0]:
# ============================================================
# CELDA 15 - COMPARACIÓN PRÁCTICA ENTRE PYSPARK Y SQL
# ============================================================

# Cómo se lee:
#   Importamos funciones para medir tiempo y para crear una tabla comparativa.
# Para qué sirve:
#   Esto permite comparar, dentro del mismo notebook, cómo se resuelve
#   una consulta equivalente usando PySpark y usando SQL.

from time import perf_counter
from pyspark.sql.functions import col, count, avg, round as spark_round

print("=" * 70)
print("COMPARACIÓN PRÁCTICA ENTRE PYSPARK Y SQL")
print("=" * 70)

# Cómo se lee:
#   Definimos el nombre completo de la tabla creada en Databricks.
# Para qué sirve:
#   Permite consultar la tabla Delta sin escribir su nombre completo varias veces.

nombre_tabla = "workspace.ea3_bigdata.titanic_pasajeros"

# Cómo se lee:
#   Cargamos la tabla como DataFrame de Spark.
# Para qué sirve:
#   Esta será la base para ejecutar la consulta con PySpark.

df_titanic = spark.table(nombre_tabla)

print("\n" + "=" * 70)
print("1. CONSULTA EQUIVALENTE USANDO PYSPARK")
print("=" * 70)

# Cómo se lee:
#   Tomamos la hora inicial antes de ejecutar la consulta con PySpark.
# Para qué sirve:
#   Permite calcular una medición sencilla del tiempo de ejecución.

inicio_pyspark = perf_counter()

# Cómo se lee:
#   Agrupamos por clase del tiquete y calculamos conteo, supervivencia y tarifa promedio.
# Para qué sirve:
#   Esta operación muestra cómo PySpark permite construir análisis mediante DataFrames.

resultado_pyspark = (
    df_titanic
    .groupBy("Pclass")
    .agg(
        count("*").alias("total_pasajeros"),
        spark_round(avg("Survived") * 100, 2).alias("porcentaje_supervivencia"),
        spark_round(avg("Fare"), 2).alias("tarifa_promedio")
    )
    .orderBy("Pclass")
)

# Cómo se lee:
#   Mostramos el resultado de la consulta PySpark.
# Para qué sirve:
#   Deja evidencia visible en el notebook.

display(resultado_pyspark)

# Cómo se lee:
#   Calculamos el tiempo final de ejecución de la consulta.
# Para qué sirve:
#   Permite registrar una referencia básica de rendimiento.

tiempo_pyspark = perf_counter() - inicio_pyspark


print("\n" + "=" * 70)
print("2. CONSULTA EQUIVALENTE USANDO SPARK SQL")
print("=" * 70)

# Cómo se lee:
#   Tomamos la hora inicial antes de ejecutar la consulta SQL.
# Para qué sirve:
#   Permite comparar el tiempo de ejecución frente a la consulta PySpark.

inicio_sql = perf_counter()

# Cómo se lee:
#   Ejecutamos una consulta SQL equivalente a la consulta construida en PySpark.
# Para qué sirve:
#   Muestra que Spark SQL puede procesar la misma tabla usando lenguaje declarativo.

resultado_sql = spark.sql("""
SELECT
    Pclass,
    COUNT(*) AS total_pasajeros,
    ROUND(AVG(Survived) * 100, 2) AS porcentaje_supervivencia,
    ROUND(AVG(Fare), 2) AS tarifa_promedio
FROM workspace.ea3_bigdata.titanic_pasajeros
GROUP BY Pclass
ORDER BY Pclass
""")

# Cómo se lee:
#   Mostramos el resultado obtenido con SQL.
# Para qué sirve:
#   Deja evidencia visible de la consulta declarativa en el notebook.

display(resultado_sql)

# Cómo se lee:
#   Calculamos el tiempo final de ejecución de la consulta SQL.
# Para qué sirve:
#   Permite registrar una referencia básica de rendimiento.

tiempo_sql = perf_counter() - inicio_sql


print("\n" + "=" * 70)
print("3. TABLA COMPARATIVA DE EXPERIENCIA DE USO")
print("=" * 70)

# Cómo se lee:
#   Creamos una lista con los criterios comparativos entre SQL y PySpark.
# Para qué sirve:
#   Esta tabla resume la experiencia de la práctica y ayuda a cumplir la rúbrica.

comparacion = [
    {
        "criterio": "Facilidad de lectura",
        "SQL": "Más sencillo para consultas directas, filtros, conteos y agrupaciones.",
        "PySpark": "Más técnico, pero permite construir procesos paso a paso."
    },
    {
        "criterio": "Transformación de datos",
        "SQL": "Adecuado para transformaciones simples o consultas declarativas.",
        "PySpark": "Más flexible para limpieza, preparación y transformación avanzada."
    },
    {
        "criterio": "Uso en notebooks",
        "SQL": "Muy útil para validar rápidamente tablas y resultados.",
        "PySpark": "Muy útil para combinar lógica de programación con procesamiento distribuido."
    },
    {
        "criterio": "Curva de aprendizaje",
        "SQL": "Más cercano a bases de datos tradicionales.",
        "PySpark": "Requiere comprender DataFrames, funciones y encadenamiento de operaciones."
    },
    {
        "criterio": "Escenarios recomendados",
        "SQL": "Consultas exploratorias, reportes y agregaciones simples.",
        "PySpark": "Pipelines, validaciones, analítica avanzada y preparación de datos."
    }
]

# Cómo se lee:
#   Convertimos la lista anterior en una tabla de Spark.
# Para qué sirve:
#   Permite mostrar la comparación como evidencia visible en Databricks.

df_comparacion = spark.createDataFrame(comparacion)

display(df_comparacion)


print("\n" + "=" * 70)
print("4. RESUMEN DE TIEMPOS DE EJECUCIÓN")
print("=" * 70)

# Cómo se lee:
#   Creamos una tabla con los tiempos aproximados de ejecución.
# Para qué sirve:
#   Deja una evidencia complementaria de comparación entre ambos enfoques.

resumen_tiempos = [
    {
        "enfoque": "PySpark DataFrame",
        "tiempo_aproximado_segundos": round(tiempo_pyspark, 4),
        "observacion": "Consulta construida mediante operaciones encadenadas sobre DataFrames."
    },
    {
        "enfoque": "Spark SQL",
        "tiempo_aproximado_segundos": round(tiempo_sql, 4),
        "observacion": "Consulta declarativa ejecutada directamente sobre la tabla Delta."
    }
]

df_tiempos = spark.createDataFrame(resumen_tiempos)

display(df_tiempos)

print("\n" + "=" * 70)
print("COMPARACIÓN FINALIZADA")
print("=" * 70)
print("Se compararon SQL y PySpark usando una consulta equivalente sobre la misma tabla Delta.")
print("La evidencia muestra diferencias de uso, lectura, flexibilidad y aplicación práctica.")

COMPARACIÓN PRÁCTICA ENTRE PYSPARK Y SQL

1. CONSULTA EQUIVALENTE USANDO PYSPARK


Pclass,total_pasajeros,porcentaje_supervivencia,tarifa_promedio
1,216,62.96,84.15
2,184,47.28,20.66
3,491,24.24,13.68



2. CONSULTA EQUIVALENTE USANDO SPARK SQL


Pclass,total_pasajeros,porcentaje_supervivencia,tarifa_promedio
1,216,62.96,84.15
2,184,47.28,20.66
3,491,24.24,13.68



3. TABLA COMPARATIVA DE EXPERIENCIA DE USO


PySpark,SQL,criterio
"Más técnico, pero permite construir procesos paso a paso.","Más sencillo para consultas directas, filtros, conteos y agrupaciones.",Facilidad de lectura
"Más flexible para limpieza, preparación y transformación avanzada.",Adecuado para transformaciones simples o consultas declarativas.,Transformación de datos
Muy útil para combinar lógica de programación con procesamiento distribuido.,Muy útil para validar rápidamente tablas y resultados.,Uso en notebooks
"Requiere comprender DataFrames, funciones y encadenamiento de operaciones.",Más cercano a bases de datos tradicionales.,Curva de aprendizaje
"Pipelines, validaciones, analítica avanzada y preparación de datos.","Consultas exploratorias, reportes y agregaciones simples.",Escenarios recomendados



4. RESUMEN DE TIEMPOS DE EJECUCIÓN


enfoque,observacion,tiempo_aproximado_segundos
PySpark DataFrame,Consulta construida mediante operaciones encadenadas sobre DataFrames.,0.9774
Spark SQL,Consulta declarativa ejecutada directamente sobre la tabla Delta.,1.0574



COMPARACIÓN FINALIZADA
Se compararon SQL y PySpark usando una consulta equivalente sobre la misma tabla Delta.
La evidencia muestra diferencias de uso, lectura, flexibilidad y aplicación práctica.


## Análisis comparativo final

Los resultados obtenidos muestran que SQL y PySpark no deben entenderse como herramientas excluyentes, sino como formas complementarias de trabajar sobre la misma infraestructura de datos. En la consulta comparativa se obtuvo el mismo análisis por clase del tiquete, lo cual confirma que ambos enfoques permiten procesar la tabla Delta creada en Databricks.

SQL fue más cómodo para expresar la consulta de forma directa: seleccionar columnas, agrupar por clase, contar pasajeros y calcular promedios. Esta forma de trabajo es útil cuando el objetivo principal es consultar, explorar o resumir información. Además, facilita la lectura para personas que ya conocen bases de datos relacionales.

PySpark, por su parte, permitió construir el análisis mediante operaciones encadenadas sobre un DataFrame. Aunque su escritura puede ser un poco más extensa, ofrece mayor control para procesos de transformación, validación, limpieza y preparación de datos. Esta ventaja es importante en escenarios de Big Data, donde los datos no solo se consultan, sino que también deben ser depurados, transformados y preparados para análisis posteriores.

En conclusión, para esta práctica SQL resultó más simple y rápido de interpretar en validaciones puntuales, mientras que PySpark ofreció una estructura más flexible para desarrollar procesos de procesamiento de datos dentro de una infraestructura cloud basada en Databricks y Apache Spark.

In [0]:
# ============================================================
# CELDA FINAL - MATRIZ DE CUMPLIMIENTO DE LA RÚBRICA EA3
# ============================================================

# Cómo se lee:
#   Esta celda construye una matriz de verificación final.
# Para qué sirve:
#   Permite demostrar, dentro del notebook, que la actividad cumple
#   con los criterios principales solicitados en la rúbrica.

print("=" * 70)
print("MATRIZ FINAL DE CUMPLIMIENTO - EA3 BIG DATA")
print("=" * 70)

# Cómo se lee:
#   Definimos una lista con los criterios de la rúbrica.
# Para qué sirve:
#   Cada fila resume qué se hizo y en qué parte del notebook queda evidenciado.

cumplimiento_rubrica = [
    {
        "criterio": "Diseño del esquema de datos",
        "puntaje": "20 pts",
        "cumplimiento": "Cumplido",
        "evidencia_en_notebook": "Se definió el esquema Titanic con StructType, DDL Spark SQL y diccionario de datos.",
        "observacion": "Incluye tipos de datos, llaves, nulabilidad y descripción de campos."
    },
    {
        "criterio": "Configuración y despliegue en Databricks CE",
        "puntaje": "20 pts",
        "cumplimiento": "Cumplido",
        "evidencia_en_notebook": "Se imprimió versión de Python, sistema operativo, versión de Spark, catálogo, esquema y entorno serverless.",
        "observacion": "La configuración fue consultada sin usar sparkContext, por compatibilidad con Databricks Serverless."
    },
    {
        "criterio": "Obtención e ingestión de datos",
        "puntaje": "15 pts",
        "cumplimiento": "Cumplido",
        "evidencia_en_notebook": "Se cargó el dataset Titanic desde una fuente pública equivalente al dataset de Kaggle y se creó una tabla Delta.",
        "observacion": "La tabla quedó disponible como workspace.ea3_bigdata.titanic_pasajeros."
    },
    {
        "criterio": "Validaciones con Spark y SQL",
        "puntaje": "15 pts",
        "cumplimiento": "Cumplido",
        "evidencia_en_notebook": "Se ejecutaron validaciones de esquema, conteos, descripción estadística, SELECT, GROUP BY y filtros.",
        "observacion": "Las validaciones fueron desarrolladas tanto con PySpark como con Spark SQL."
    },
    {
        "criterio": "Comparación entre SQL y Spark",
        "puntaje": "15 pts",
        "cumplimiento": "Cumplido",
        "evidencia_en_notebook": "Se compararon ambos enfoques mediante consulta equivalente, tabla comparativa y análisis final.",
        "observacion": "Se identificaron ventajas, limitaciones y escenarios recomendados para cada enfoque."
    },
    {
        "criterio": "Entrega en GitHub",
        "puntaje": "5 pts",
        "cumplimiento": "Pendiente de exportar y subir",
        "evidencia_en_notebook": "El notebook debe exportarse como Gomez_Pablo_Actividad_2.ipynb y subirse al repositorio Actividad_2.",
        "observacion": "Este paso se realiza al finalizar la edición del notebook."
    },
    {
        "criterio": "Video de sustentación",
        "puntaje": "10 pts",
        "cumplimiento": "Pendiente de grabar",
        "evidencia_en_notebook": "El notebook contiene la ruta completa para explicar el paso a paso en el video.",
        "observacion": "El video debe mostrar configuración, esquema, ingesta, tabla, validaciones y comparación SQL vs Spark."
    }
]

# Cómo se lee:
#   Convertimos la lista anterior en un DataFrame de Spark.
# Para qué sirve:
#   Permite mostrar la matriz de cumplimiento como tabla visible dentro de Databricks.

df_cumplimiento = spark.createDataFrame(cumplimiento_rubrica)

display(df_cumplimiento)

print("\n" + "=" * 70)
print("RESUMEN FINAL")
print("=" * 70)
print("El notebook cubre el diseño del esquema, la configuración de Databricks,")
print("la ingesta de datos, la creación de la tabla Delta, las validaciones")
print("con PySpark y SQL, y la comparación entre ambos enfoques.")
print("Los pasos pendientes son externos al notebook: exportar el archivo, subirlo")
print("a GitHub y grabar el video de sustentación.")

MATRIZ FINAL DE CUMPLIMIENTO - EA3 BIG DATA


criterio,cumplimiento,evidencia_en_notebook,observacion,puntaje
Diseño del esquema de datos,Cumplido,"Se definió el esquema Titanic con StructType, DDL Spark SQL y diccionario de datos.","Incluye tipos de datos, llaves, nulabilidad y descripción de campos.",20 pts
Configuración y despliegue en Databricks CE,Cumplido,"Se imprimió versión de Python, sistema operativo, versión de Spark, catálogo, esquema y entorno serverless.","La configuración fue consultada sin usar sparkContext, por compatibilidad con Databricks Serverless.",20 pts
Obtención e ingestión de datos,Cumplido,Se cargó el dataset Titanic desde una fuente pública equivalente al dataset de Kaggle y se creó una tabla Delta.,La tabla quedó disponible como workspace.ea3_bigdata.titanic_pasajeros.,15 pts
Validaciones con Spark y SQL,Cumplido,"Se ejecutaron validaciones de esquema, conteos, descripción estadística, SELECT, GROUP BY y filtros.",Las validaciones fueron desarrolladas tanto con PySpark como con Spark SQL.,15 pts
Comparación entre SQL y Spark,Cumplido,"Se compararon ambos enfoques mediante consulta equivalente, tabla comparativa y análisis final.","Se identificaron ventajas, limitaciones y escenarios recomendados para cada enfoque.",15 pts
Entrega en GitHub,Pendiente de exportar y subir,El notebook debe exportarse como Gomez_Pablo_Actividad_2.ipynb y subirse al repositorio Actividad_2.,Este paso se realiza al finalizar la edición del notebook.,5 pts
Video de sustentación,Pendiente de grabar,El notebook contiene la ruta completa para explicar el paso a paso en el video.,"El video debe mostrar configuración, esquema, ingesta, tabla, validaciones y comparación SQL vs Spark.",10 pts



RESUMEN FINAL
El notebook cubre el diseño del esquema, la configuración de Databricks,
la ingesta de datos, la creación de la tabla Delta, las validaciones
con PySpark y SQL, y la comparación entre ambos enfoques.
Los pasos pendientes son externos al notebook: exportar el archivo, subirlo
a GitHub y grabar el video de sustentación.
